# Day 6 Assignment — Gold Layer

## Task 3 — Create Gold Table

The Gold layer contains business-ready aggregated data created from the Silver layer.

Since the dataset does not contain store information, a daily revenue report is created using order_date.

The Gold table contains:

- Total Revenue
- Total Orders
- Total Quantity Sold
- Total Discount

Target table:

`dev.gold.daily_revenue`

In [0]:
from pyspark.sql.functions import sum, countDistinct, round

silver_df = spark.table("dev.silver.sales_clean1")

display(silver_df)

### Daily Revenue

In [0]:
gold_df = (
    silver_df
    .groupBy("order_date")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("discount_amount"), 2).alias("total_discount"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("order_date")
)

display(gold_df)

In [0]:
gold_df.write.mode("overwrite").saveAsTable("dev.gold.daily_revenue")

### Revenue by Product

In [0]:
product_revenue_df = (
    silver_df
    .groupBy("product_id")
    .agg(
        round(sum("total_amount"), 2).alias("total_revenue"),
        sum("quantity").alias("total_quantity")
    )
    .orderBy("total_revenue", ascending=False)
)

## Gold Layer Result

The Silver sales data was transformed into a business-ready Gold table.

The Gold table provides:

- Daily revenue
- Total orders
- Total quantity sold
- Total discounts

The aggregation is performed using `order_date`.

Target table:

`dev.gold.daily_revenue`

This table can be used directly by analysts, dashboards, and reporting tools.

### Product Performance Gold Table

# Task 7 — Second Gold Aggregation

## Product Performance Summary

A second Gold aggregation is created for the Product/Management team.

The aggregation summarizes sales performance by product, including:

- Total quantity sold
- Total revenue
- Total orders
- Average order value

Target table:

`dev.gold.product_performance`

In [0]:
from pyspark.sql.functions import (
    sum,
    countDistinct,
    avg,
    round
)

silver_df = spark.table("dev.silver.sales_clean1")

display(silver_df)

In [0]:
product_performance_df = (
    silver_df
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        round(avg("total_amount"), 2).alias("average_order_value")
    )
    .orderBy("total_revenue", ascending=False)
)

display(product_performance_df)

In [0]:
product_performance_df.write.mode("overwrite").saveAsTable("dev.gold.product_performance")

## Why This Aggregation Belongs in Gold ?

The product performance aggregation belongs in the Gold layer because it is a business-ready metric that can be reused by multiple consumers such as product managers, business analysts, and management.

If every stakeholder calculated total revenue, quantity sold, and order counts independently from the Silver layer, different teams could use different filters or calculation logic and produce inconsistent results.

By calculating and storing these metrics in Gold, the business gets a consistent and governed definition of product performance.

The Gold table also avoids requiring analysts to repeatedly perform the same aggregation on cleaned Silver data. This improves reusability and makes the data easier to consume for dashboards and reporting.

## Primary Stakeholder

**Product Management / Business Analytics Team**

The team can use this table to identify:

- Best-performing products
- Products generating the highest revenue
- Products with the highest sales volume
- Average order value associated with products